Import das bibliotecas necessárias

In [1]:

# Import da classe Document da biblioteca docx que tem como objetivo criar
# objetos que representam um documento word.
from docx import Document

# Import da biblioteca pandas de análise de dados que tem como objetivo acessar 
# valores presentes em arquivos csv, xlsx, entre outros.
import pandas as pd

# Import da biblioteca os que tem como objetivo possibilitar que o python se 
# comunique com o sistema de arquivos do sistema operacional 
import os

# Biblioteca que contém a função move que realiza a transferencia de arquivos
# de um local para o outro.
import shutil

# Biblioteca que ira configurar o servidor de envio de emails.
import smtplib

# Cria o container (a caixa principal) do e-mail. Por padrão, um e-mail simples
# é apenas um texto. Quando nós precisamos enviar um e-mail que contém varias
# partes combinadas (por exemplo: um texto formatado + um arquivo em anexo),
# precisamos do MimeMultipart que junta o corpo da mensagem e os arquivos
# anexados em uma unica estrutura.     
from email.mime.multipart import MIMEMultipart

# É o módulo usado para criar o conteúdo visivel da mensagem (o corpo do -email).
# Ele permite definir se o conteúdo sera enviado em texto simples (plain) ou 
# formatado em HTML(html), garantindo que acentos e caracteres especiais fiquem
# corretos através do suporte á codificação (como utf-8).
from email.mime.text import MIMEText

# Representa os arquivos em anexo. Serve como base para preparar arquivos
# que não são texto puro (como PDFs, imagens, planilhas ou documentos.docx).
# Ele prepara o arquivo de uma maneira que os servidores de e-mail entendam 
# como um dado binário (application/octet-stream).
from email.mime.base import MIMEBase

# Converte o arquivo anexo em texto seguro para envio. Os servidores
# de e-mail foram originalmente projetados para trafegar apenas texto
# simples. O módulo encoders (geralmente usado na função encoders.encode_base64)
# codifica o arquivo binário (nosso PDF) para o formato Base64. Isso transforma
# os dados binários do arquivo em uma sequência segura de caracteres, garantindo
# que o anexo não seja corrompido ou bloqueado pelos servidores SMTP durante
# o envio. 
from email import encoders

# Biblioteca que possibilita a manipulação de datas 
import datetime

# Função da biblioteca docx2pdf que tem como objetivo converter arquivos
# para o formato PDF 
from docx2pdf import convert

# Funçãp da biblioteca dotenv que tem como objetivo carregar variáveis de
# ambientes no nosso código.
from dotenv import load_dotenv

C:\Users\caike\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Criaçao das pastas necessárias para o projeto

In [2]:
# Makedirs: Função da biblioteca os que tem como objetivo criar diretórios no projeto.
# A função recebe como argumento o caminho que devera conter a pasta criada e o exist_ok
# que indica que a função ira ignorar a pasta sem dar erros no código, caso a pasta ja exista
# no sistema
os.makedirs("modelo", exist_ok=True )

os.makedirs("Planilhas", exist_ok=True)

os.makedirs("Processados", exist_ok=True)

os.makedirs("Contratos", exist_ok=True)

os.makedirs("Erros", exist_ok=True)

os.makedirs("PDFs", exist_ok=True)

# Ira indicara criação da pasta
print("Pastas criadas no sistema:")

# listdir: Função da biblioteca os que tem como objetivo listar arquivos e diretórios de uma pasta.
# A função pode receber como argumento o caminho da pasta que terá os arquivos listados.
# Observação: Caso você não especifique um caminho, ele irá listar os arquivos e pastas presentes
# no diretório do seu projeto
lista_pasta_criadas = os.listdir()

# Como a nossa ideia é mostrar apenas as pastas do projeto (de preferencia, as que foram criadas
# pela automação), vamos filtrar os itens presentes no diretório do projeto

# Primeiro, vamos percorrer a lista de pastas 
for pasta in lista_pasta_criadas:

    # Como o objetivo é mostrar apenas pastas, vamos acessar apenas itens que não contém
    # o '.', pois, pastas não possuem extensão.
    if '.' not in pasta:
        
        # Irá imprimir todas as pastas.
        print(pasta)



Pastas criadas no sistema:
Contratos
Erros
LICENSE
modelo
PDFs
Planilhas
Processados


Criando o caminho de acesso para a nossa pasta de planilhas

In [4]:
# Ira listar todos os arquivos da pasta planilhas
pasta_planilhas = os.listdir('Planilhas')

pasta_planilhas

['exercicio d de bd - Copia.brM',
 'Lista 1 de candidatos.xlsx',
 'Lista 2 de candidatos.xlsx']

caminho do modelo gerado pelo RH

In [6]:
modelo_rh = os.listdir('modelo') 

# Ira listar os modelos de contratos criados pelo RH
modelo_rh

['modelo contratação.docx']

Criando a função de email

In [7]:
# Ira carregar as variáveis de ambiente no código
load_dotenv()

# getenv: Função que acessa os valores das variáveis de ambientes.
EMAIL = os.getenv('EMAIL')

SENHA = os.getenv('SENHA')

# Função que irá enviar os contratos via email. Ela recebe como argumento o email do candidato,
# o nome do candidato, o caminho do arquivo que será enviado e a planilha que contém o dado do 
# candidato.
def enviar_email(email_candidato, nome_candidato, caminho_pdf, planilha_acessada):

   # Irá inspecionar o bloco de código com o objetivo de capturar possiveis erros de execução do código
   try:

         # Endereço do servidor de email do gmail
         smtp_server = "smtp.gmail.com"

         # Porta que o servidor irá utilizar para transferir informações
         port = 587

         # email que enviara os contratos 
         email_remetente = EMAIL

         # Senha de app do email que será utilizado para realizar os envios de contrato
         senha_app = SENHA

         # Instancia da classe MIMEMultipart que irá criar o conteudo da nossa mensagem
         mensagem = MIMEMultipart()

         # ['FROM']: Atributo da classe MIMEMultipart que recebe como valor
         # o email do remetente
         mensagem['From'] = email_remetente

         # ['To']: Atributo da classe MIMEMultipart que recebe como valor
         # o email do destinatario.
         mensagem['To'] = email_candidato

         # ['Subject']: Atributo da classe MIMEMultipart que recebe como valor
         # o assunto do e-mail.
         mensagem['Subject'] = f"Contrato de Trabalho - {nome_candidato}"

         # Ira conter o conteudo da mensagem que iremos enviar ao destinatário.
         corpo = f"Olá, {nome_candidato} segue em anexo o seu contrato de trabalho\n"

         # attach: Método da classe MIMEMultipart que tem como objetivo montar 
         # a nossa mensagem usando o MIMEText que define como o conteudo da
         # mensagem sera montado.  
         mensagem.attach(MIMEText(corpo, 'plain', 'utf-8'))

         # Nessa etapa iremos verificar se o caminho do arquivo que será anexado
         # existe.

         # path: Módulo da biblioteca os que possui funções que manipulam arquivos

         # exists: Função do módulo path que tem como objetivo verificar a existência
         # de caminhos/arquivos. A função recebe como argumento o caminho do arquivo
         # que esta sendo verificado.
         if os.path.exists(caminho_pdf):
            
            # with: Comando que realiza o fechamento do arquivo acessado após
            # a execução da função open, dessa forma evitamos que arquivos
            # fiquem corrompidos

            # open: Função nativa do python que tem como objetivo criar, editar
            # e abrir arquivos.

            # caminho_pdf: Caminho do arquivo PDF que será enviado.

            # rb(read binary): Indica que o arquivo será apenas de leitura
            # (não podera ser editado) e será lido no formato binário e não
            # como um texto comum

            # anexo: Representa o arquivo que esta sendo aberto 
            with open(caminho_pdf, "rb") as anexo:
               
                # Cria a estrutura inicial para o anexo. O tipo "application"
                # com o subtipo "octet-stream" é o padrão MIME universal para
                # indicar que o conteúdo é um fluxo arbitrario de bytes brutos
                # (ou seja, um arquivo genérico que não é texto, como PDF, ZIP
                # ou executavel). Isso avisa ao e-mail que aquilo deve ser tratado
                # como um download/anexo.
                base = MIMEBase("application", "octet-stream")
                
                # Insere o conteúdo binário do arquivo dentro do objeto base.
                # O anexo.read() lê todos os bytes PDF (que foi aberto com o
                # "rb") e o método set_payload() carrega esses dados brutos
                # para dentro do container do anexo.
                base.set_payload(anexo.read())
               
                # Converte os dados do arquivo para a codificação Base64.
                #  Os protocolos de envio de e-mail (como o SMTP) funcionam
                # nativamente transportando caracteres de texto. Se você tentar
                # enviar bytes brutos de um PDF direto, eles serão corrompidos.
                # A função transforma o arquivo binário em uma sequência de 
                # caracteres de texto ASCII seguros, garantindo que o PDF chegue
                # intacto ao destinatário.
                encoders.encode_base64(base)
                
                # Define como o e-mail deve exibir o arquivo e qual nome ele terá.

                # Content-Disposition: attachment: Indica ao leitor de e-mail (Gmail,
                # Outlook, etc) que este arquivo deve aparecer como um anexo baixavel,
                # e não exibido no meio do texto.
                
                # filename: Define o nome exato que o arquivo terá ao ser recebido.

                # os.path.basename(caminho_pdf): extrai apenas o nome do arquivo
                # (ex: contrato_maria.pdf), removendo o caminho das pastas (ex: PDFs/contrato_maria.pdf)
                base.add_header("Content-Disposition", f"attachment; filename={os.path.basename(caminho_pdf)}")
                
                # Anexa o objeto do arquivo (já processado, codificado e identificado)
                # á mensagem principal (MIMEMultipart). Junta a estrutura do anexo que
                # acabamos de montar ao "envelope" da mensagem onde ja estão o remetente,
                # destinatario, assunto e o corpo do e-mail.
                mensagem.attach(base)

            # Instância da classe SMTP que ira conter as informações do servidor
            # de envio de e-mail. A classe recebe em seu construtor o endereço
            # do servidor e a porta que será utilizada.             
            server = smtplib.SMTP(smtp_server, port)

            # Método da classe SMTP que protege as informações da mensagem que
            # esta sendo enviada  
            server.starttls()

            # Método da classe SMTP que irá se conectar com a conta de e-mail do
            # remetente que enviara as mensagens. O método recebe como argumento
            # o email e a senha do remetente  
            server.login(email_remetente, senha_app)

            # Metodo da classe SMTP que enviará as mensagens. O método recebe
            # como argumento a mensagem que será enviada.
            server.send_message(mensagem)

            # Ira encerrar a operação/execução do servidor após os envios das 
            # mensagens
            server.quit()
        
         else:
            # Se o caminho não for encontrado, vamos executar esse trecho que irá
            # registrar logs de erro relacionados a envio de e-mails. 
            print(f"O caminho {caminho_pdf} não foi encontrado")

            # encoding: Adapta o código para entender caracteres especiais no
            # texto da mensagem

            # a: comando append que adiciona no arquivo os valores escritos no
            # write.
            with open(f"Erros/erro_email_{planilha_acessada}.txt", "a", encoding='utf-8') as erro_log:
               
               # write: Método que irá escrever valores no arquivo
                erro_log.write(f"Não foi possivel enviar o email para o candidato {nome_candidato} pois o seu contrato em PDF não foi encontrado\n")
        
   except Exception as erro:
         # Exceção que irá lidar com a falha de envio de e-mails
        print(f"Falha no envio do email: {erro}")

Acessando a pasta de planilhas e construindo contratos

In [8]:

# Laço que irá percorrer os arquivos da pasta de planilhas e irá realizar
# o processo de emissão de contratos e envio de e-mails.
for arquivo in pasta_planilhas:

  # Ira inspecionar o bloco de código com o objetivo de capturar possiveis erros
  # de execução.  
  try:
      # Ira verificar se a extensão do arquivo acessado é do tipo 'xlsx'
      # (formato excel). 
      
      # endswith: Função nativa do python que tem como objetivo verificar
      # os caracteres finais de um texto. A função recebe como argumento
      # os caracteres que queremos verificar no final do texto.
      if arquivo.endswith('.xlsx'):
         # Se essa condição for verdadeira, ou seja, se o arquivo acessado for
         # um excel, vamos iniciar de acesso aos dados, emissão de contratos e
         # envio de e-mails.

         # Como precisamos acessar os valores através dos nomes das colunas, vamos padronizar os nomes das colunas com o 
         # o objetivo de evitar pequenas alterações nos nomes das colunas.  
         nomes_colunas = ["nome", "cpf", "endereco", "cargo", "salario", "cidade-uf", "data de inicio", "email"]

        # read_excel: Função do pandas que tem como objetivo carregar na memória dados presentes em arquivos
        # excel. Ela retona um objeto DataFrame que contém metodos e atributos para manipulação de datasets

        # planilhas/arquivo: Caminho da planilha que será acessada.

        # header=0: Indica que o código deve considerar como coluna a primeira linha
        # de valores encontrada
        
        # names: Substitui os nomes da primeira linha pelos nomes definidos na lista
        # de nomes de coluna.      
         df_planilhas = pd.read_excel(f"Planilhas/{arquivo}", header=0, names=nomes_colunas)

        # Após carregar os dados, vamos tratar os cpfs para garantirmos que os dados
        # estejam com o formato correto para serem utilizados

        # Primeiro, vamos substituir os valores nulos da coluna de cpf por valores vázios
        # fillna: Metodo da classe retornada pelo read_excel que tem como objetivo substituir
        # valores nulos por outros valores. A função recebe como argumento o valor que irá
        # substituir os valores nulos.
         cpf = df_planilhas['cpf'].fillna("")
         
         # astype: Metodo que irá converter os dados para o tipo string, pois,
         # precisamos processar os caracteres especiais presentes em CPFs
         cpf_limpo = cpf.astype(str, errors='ignore')

         # Lista que irá conter os cpfs que estão com formato inválido, vamos
         # usar essa lista como parametro para filtragem de cpfs válidos.
         cpf_invalidos = []
        
         # Laço que ira percorrer cada cpf limpo presente no dataset carregado
         # na memória.
         for cpf_candidato in cpf_limpo:

            # Basicamente essa variável irá comparar se o cpf acessado esta presente
            # em mais de uma linha. Vamos utilizar o sum para somar a quantidade de 
            # cpfs iguais encontrados no dataset. 
            total_cpf = (cpf_limpo == cpf_candidato).sum()
            
            # Ira verificar se a quantidade de caracteres é diferente de 14.
            # Apesar do cpf conter 11 digitos, eu defini 14 por que nesse projeto
            # precisamos utilizar os caracteres especiais (.,-) que totaliza 14
            # caracteres no cpf acessado.
            if len(cpf_candidato) != 14:

                # Se essa condição for verdadeira, vamos iniciar a construção do
                # log de erro que indica ao usuário os problemas do cpf acessado.

                # with: Comando que possibilita que o código feche o arquivo após
                # a execução do código.
                
                # open: Função nativa do python que tem como objetivo acessar, editar
                # e criar arquivos.  

                # "Erros/erros_{arquivo}.txt": Caminho que irá conter o arquivo criado/editado

                # "a": Comando append que insere dados e valores em um arquivo. Observação:
                # se usarmos o comando w, o código irá sobrescrever o conteudo inserido
                # anteriormente.

                # encoding: Adapta o código para interpretar caracteres especiais.

                # erro_log: Representa o arquivo que será criado/editado.
                with open(f"Erros/erros_{arquivo}.txt", "a", encoding='utf-8') as erro_log:
                  
                  # write: Comando que ira gravar os valores inseridos pelo append. A função recebe
                  # como argumento o texto que será gravado no arquivo.
                  erro_log.write(f"O cpf {cpf_candidato} da planilha {arquivo} não possui 11 digitos\n")

                  # Vamos adicionar o cpf invalido na lista de cpfs inválidos.
                  cpf_invalidos.append(cpf_candidato)

            # Ira verificar se a soma de cpfs iguais é maior que 1, o que indica
            # a existência de cpfs iguais no dataset carregado na memória.
            elif total_cpf > 1:
              
              # Se essa condição for verdadeira, vamos iniciar o mesmo processo feito
              # no if anterior 
              with open(f"Erros/erros_{arquivo}.txt", "a", encoding='utf-8') as erro_log:

                erro_log.write(f"O {cpf_candidato} da planilha {arquivo} está presente em mais de um registro\n")

                cpf_invalidos.append(cpf_candidato)

         # Após realizar as validações e criar os logs vamos iniciar a filtragem
         # de cpfs válidos e a criação dos contratos.

         # Ira filtrar os cpfs que são diferentes dos cpfs presentes na lista
         # de inválidos. 
         # isin: O método isin() verifica se os elementos estão na lista. O símbolo de til (~) na frente da expressão é o operador lógico NOT,
         #  que inverte a seleção. Portanto, o trecho  significa "filtre as linhas cujo CPF NÃO está na lista de inválidos"  
         df_cpfs_validos = df_planilhas[~cpf_limpo.isin(cpf_invalidos)]  
         
         # Ira percorrer a variável de cpfs validos usando o método iterrows
         # que serve para percorrer o datafrmae linha por linha utilizando
         # indices (posição dos dados). Esse método possibilta o acesso a dados
         # através do nome das colunas.
         for index, linha in df_cpfs_validos.iterrows():

            # Instancia da classe Document que cria os documentos word. A classe
            # recebe como argumento em seu construtor o caminho do arquivo que será
            # aberto no código (o modelo padrão de contratos). 
            documento = Document(f"modelo/{modelo_rh[0]}")

            # Ira conter os dados de cada candidato presente na filtragem de cpfs válidos.
            # o str ira converter todos os dados para string, dessa forma evitamos erros de
            # tipo de dados/formatos. 
            nome = str(linha['nome'])

            cpf = str(linha['cpf'])

            endereco = str(linha['endereco'])

            cargo = str(linha['cargo'])

            salario = str(linha['salario'])

            cidade_uf = str(linha['cidade-uf'])

            # Ira coletar e formatar a data atual do sistema (data de emissão do contrato).
            data_emissao = datetime.datetime.now().strftime("%d/%m/%Y")

            data_inicio = str(linha['data de inicio'])

            email = (linha['email'])

            # Dicionário que ira conter os itens do contrato com os dados do candidato. Vamos usar essa lista
            # para substituir o nome das chaves pelo nome do valor atribuido a chave.
            itens_contratos = {'[NOME_FUNCIONARIO]': nome, 
                              '[CPF_FUNCIONARIO]': cpf, 
                              '[ENDERECO_FUNCIONARIO]': endereco,
                              '[CARGO_FUNCIONARIO]':cargo, 
                              '[SALARIO_FUNCIONARIO]':salario, 
                              '[DATA_INICIO]':data_inicio,
                              '[CIDADE-UF]': cidade_uf,
                              '[DATA_EMISSÃO]': data_emissao}

            # Caso o valor acessado seja igual ao nome da coluna.
            if nome == 'nome' and cpf == 'cpf' and endereco == 'endereco' and cargo == 'cargo' and salario == 'salario' and data_inicio == 'data de inicio' and cidade_uf == 'cidade-uf':
                
                # Se essa condição for verdadeira, vamos dar um continue que ignorara esses dados e não criara o contrato.
                continue                  
            
            # Ira percorrer o atributo paragraphs da classe Document que contém o conteudo do
            # documento word
            for paragrafo in documento.paragraphs:

                # Ira percorrer a lista de itens de contrato item por item (utilizando o método
                # items que possibilita acessar cada chave e seu respectivo valor).
               for chave, valor in itens_contratos.items():

                  # Ira verificar se a chave esta presente no texto (atributo text da classe Document)
                  #  do paragrafo acessado
                  if chave in paragrafo.text:

                    # Se essa condição for verdadeira, vamos subtituir o nome da chave pelo
                    # seu valor usando a função nativa replace
                    paragrafo.text = paragrafo.text.replace(chave, valor)
                         
            # Após todo esse processo, vamos usar o método save que tem como objetivo salvar os arquivos.
            # A função recebe como argumento o caminho que o arquivo será salvo.
            documento.save(f"Contratos/contrato_{nome}.docx")
            
            # Função que tem como objetivo converter arquivos para o formato PDF. A função recebe
            # como argumento o caminho do arquivo que vai ser acessado e o caminho que o arquivo
            # convertido em PDF será salvo. 
            convert(f"Contratos/contrato_{nome}.docx", f"PDFs/contrato_{nome}.pdf")

            # Chamada da função de envio de e-mails
            enviar_email(email, nome, f"PDFs/contrato_{nome}.pdf", arquivo )

          # Função da bilioteca shutil que irá transferir os arquivos ja acessados para uma outra pasta.
         shutil.move(f"Planilhas/{arquivo}", f"Processados/{arquivo}")

      else:
        
        # Se essa condição for verdadeira, vamos indicar ao usuário o arquivo que não é um excel.
        print(f"O arquivo {arquivo} não é um excel")

  except FileNotFoundError as erro:
    # Exceção que irá lidar com erros de arquivos não encontrados
    print(f"O arquivo {arquivo} não foi encontrado: {erro}")

O arquivo exercicio d de bd - Copia.brM não é um excel


100%|██████████| 1/1 [00:04<00:00,  4.16s/it]
